# Chatbot Assignment
## UCS420 - Cognitive Computing
### Medical query assistant

**Name:** Naman Sharma  
**Roll No:** 1024160039

The assistant reads a short health message, puts it in one intent, and answers from a small knowledge base. It is not a doctor. Emergency symptoms are sent to 112 before the classifier is trusted.


## Steps 1, 2 and 3: Corpus, taxonomy, and imports

Six intents:

| Intent | What belongs here |
| --- | --- |
| emergency | Chest pain, cannot breathe, collapse, one-sided weakness, seizure, heavy bleeding |
| booking | Book, move, or cancel a doctor visit |
| medication | What a common medicine is for, and the usual caution |
| location | Where a hospital, chemist, or the emergency department is |
| advice | Non-urgent home care: blood pressure, sugar, a small burn, a cold, a mild headache |
| general | Hello, thank you, and what this assistant is |

Weather, poems, and filler are not given a class. Calling them `general` would teach the model that they are normal health chat. A rule refuses them later.

Asking where the emergency department is, is `location`. The person is asking for a place, not reporting symptoms.


In [2]:
import re
import numpy as np
import nltk
from nltk import pos_tag
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

for package in ["stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger_eng"]:
    nltk.download(package, quiet=True)

taxonomy = {
    "emergency": "Symptoms that need 112 now.",
    "booking": "Book, move, or cancel a visit.",
    "medication": "General facts and cautions for a named medicine.",
    "location": "A hospital, a chemist, or the emergency department as a place.",
    "advice": "Home care when the person is not in danger.",
    "general": "A greeting or a question about this assistant.",
}

corpus = [
    ("I have a sharp pain in my chest", "emergency"),
    ("My chest hurts when I take a breath", "emergency"),
    ("Sharp chest pain started a few minutes ago", "emergency"),
    ("My chest feels tight and I am short of breath", "emergency"),
    ("I cannot breathe properly since the last few minutes", "emergency"),
    ("Sudden numbness on the left side of my body", "emergency"),
    ("My mother collapsed and is not responding", "emergency"),
    ("He is unresponsive and I cannot wake him", "emergency"),
    ("The driver collapsed and we could not wake him", "emergency"),
    ("There is chest pressure going into my left arm", "emergency"),
    ("Her speech is slurred and one side of her face is drooping", "emergency"),
    ("The bleeding will not stop and there is a lot of blood", "emergency"),
    ("Blood is pouring from his arm and it will not stop", "emergency"),
    ("She is bleeding heavily after the fall", "emergency"),
    ("He had a seizure and is not fully awake yet", "emergency"),
    ("She started having a seizure in the chair", "emergency"),
    ("My heartbeat is racing and my chest hurts", "emergency"),
    ("One arm became weak all of a sudden", "emergency"),
    ("I need to book an appointment with a doctor", "booking"),
    ("Can I schedule a visit for tomorrow morning", "booking"),
    ("How do I book a consultation with a cardiologist", "booking"),
    ("I want to change my appointment time", "booking"),
    ("Please cancel my Friday appointment", "booking"),
    ("Is a Monday morning OPD slot still free", "booking"),
    ("I want to reschedule my skin checkup", "booking"),
    ("Please book a follow-up in the medicine department", "booking"),
    ("Can the desk call me to confirm the appointment", "booking"),
    ("This is a new patient visit, not a follow-up", "booking"),
    ("Shift my evening appointment to the morning", "booking"),
    ("How early should I reach for the OPD appointment", "booking"),
    ("I want to schedule a consultation this week", "booking"),
    ("Schedule the doctor consultation for Monday", "booking"),
    ("What are the side effects of paracetamol", "medication"),
    ("How often can an adult take ibuprofen", "medication"),
    ("Does amoxicillin cause any side effects", "medication"),
    ("What should I know before taking aspirin", "medication"),
    ("Can I take a multivitamin every day", "medication"),
    ("Are antibiotics safe in pregnancy", "medication"),
    ("Which antibiotics are usually avoided in pregnancy", "medication"),
    ("Is ibuprofen okay for a small child", "medication"),
    ("What happens if someone takes too much paracetamol", "medication"),
    ("Can I take amoxicillin for a cold", "medication"),
    ("Does aspirin upset the stomach", "medication"),
    ("Should paracetamol be taken after food", "medication"),
    ("My doctor gave amoxicillin. What is it used for", "medication"),
    ("Where can I find a pharmacy near Rajindra Hospital", "location"),
    ("How do I reach a clinic in Patiala", "location"),
    ("Is the hospital emergency department open at night", "location"),
    ("Where is Rajindra Hospital", "location"),
    ("Where is the government hospital in Patiala", "location"),
    ("Is there a 24 hour chemist around the hospital", "location"),
    ("How do I get to the OPD block", "location"),
    ("Which hospital in Patiala has an emergency department", "location"),
    ("Find a clinic that is open on a Sunday", "location"),
    ("Where can I get a blood test done nearby", "location"),
    ("I need a blood test near Rajindra Hospital", "location"),
    ("Is there a hospital close to Model Town", "location"),
    ("Please share the address of the emergency room", "location"),
    ("What is the address of the OPD", "location"),
    ("How can I manage high blood pressure", "advice"),
    ("What food helps if cholesterol is high", "advice"),
    ("Tips to keep blood sugar steady", "advice"),
    ("How do I treat a small burn at home", "advice"),
    ("What daily habits help the heart", "advice"),
    ("Which habits are good for the heart", "advice"),
    ("My head hurts a little. What can I try", "advice"),
    ("What can I do about a mild headache", "advice"),
    ("A mild headache since the morning, with no fever", "advice"),
    ("My head hurts but I have no other symptom", "advice"),
    ("How do I look after a common cold at home", "advice"),
    ("I have been a bit tired today. Is rest enough", "advice"),
    ("Just tired after a long day, nothing else is wrong", "advice"),
    ("How much salt is sensible if my blood pressure is high", "advice"),
    ("What should I eat if my sugar stays high", "advice"),
    ("Does a short walk after meals help blood sugar", "advice"),
    ("What should I avoid putting on a minor burn", "advice"),
    ("Hello", "general"),
    ("Good morning", "general"),
    ("What is your name", "general"),
    ("Who are you", "general"),
    ("Thank you", "general"),
    ("Hi there", "general"),
    ("Are you a doctor or just a chatbot", "general"),
    ("Good evening", "general"),
    ("Hey", "general"),
]

print("Taxonomy:")
for name, meaning in taxonomy.items():
    print(f"  {name}: {meaning}")

labels = [label for _, label in corpus]
print("\nSentences:", len(corpus))
for name in taxonomy:
    print(f"  {name}: {labels.count(name)}")


Taxonomy:
  emergency: Symptoms that need 112 now.
  booking: Book, move, or cancel a visit.
  medication: General facts and cautions for a named medicine.
  location: A hospital, a chemist, or the emergency department as a place.
  advice: Home care when the person is not in danger.
  general: A greeting or a question about this assistant.

Sentences: 85
  emergency: 18
  booking: 14
  medication: 13
  location: 14
  advice: 17
  general: 9


## Step 4: Preprocessing

The cleaner does four things:

1. Expands contractions, so `can't breathe` stays `cannot breathe` instead of becoming the token `cant`.
2. Maps a pure greeting such as `who are you` to one token. Without this, the stopword list deletes the whole question.
3. Lemmatises with the part of speech. A word ending in `-ed` that the tagger did not mark as a verb is tried as a verb, so `collapsed` becomes `collapse`.
4. Drops English stopwords.


In [4]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

CONTRACTIONS = [
    ("can't", "cannot"),
    ("won't", "will not"),
    ("don't", "do not"),
    ("doesn't", "does not"),
    ("didn't", "did not"),
    ("i'm", "i am"),
    ("it's", "it is"),
    ("what's", "what is"),
    ("that's", "that is"),
    ("i've", "i have"),
    ("i'll", "i will"),
    ("let's", "let us"),
    ("n't", " not"),
]

GREETINGS = {
    "hello", "hi", "hey",
    "good morning", "good evening", "good afternoon",
    "thank you", "thanks", "thanks a lot",
    "who are you", "what is your name", "what are you",
    "are you a doctor", "are you a doctor or just a chatbot",
    "hi there", "nice to meet you",
}

def expand_contractions(text):
    text = text.lower().replace("’", "'")
    for source, target in CONTRACTIONS:
        text = text.replace(source, target)
    return text

def normalized(text):
    text = expand_contractions(text)
    return " ".join(re.sub(r"[^a-zA-Z\s]", " ", text).split())

def is_greeting(text):
    text = normalized(text)
    if text in GREETINGS:
        return True
    for phrase in sorted(GREETINGS, key=len, reverse=True):
        if text.startswith(phrase):
            rest = text[len(phrase):].strip(" ,")
            if rest == "" or rest in GREETINGS:
                return True
    return False

def wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("N"):
        return wordnet.NOUN
    if tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

def lemma_of(word, tag):
    lemma = lemmatizer.lemmatize(word, wordnet_pos(tag))
    if lemma == word and word.endswith("ed"):
        lemma = lemmatizer.lemmatize(word, wordnet.VERB)
    return lemma

def preprocess_text(text):
    if is_greeting(text):
        return "greeting"
    tokens = [word for word in re.sub(r"[^a-zA-Z\s]", " ", expand_contractions(text)).split() if word]
    kept = []
    for word, tag in pos_tag(tokens):
        if word in stop_words:
            continue
        kept.append(lemma_of(word, tag))
    return " ".join(kept)

print(preprocess_text("I have a sharp pain in my chest!"))
print(preprocess_text("Help, my chest hurts and I can't breathe"))
print(preprocess_text("Who are you?"))
print(preprocess_text("My mother collapsed and is not responding"))


sharp pain chest
help chest hurt cannot breathe
greeting
mother collapse respond


## Steps 5 and 6: Classifier and evaluation

Word counts go in with TF-IDF, including pairs of words, so `chest pain` is not the same feature as `head hurts`. The classifier is logistic regression.

The score below is 5-fold cross-validation. Each sentence is tested only in a fold that did not train on it. Accuracy on the training rows is not reported as the result, because that number only shows that the model remembers rows it has already seen.

After the check, the assistant is fitted once on the full labeled set.


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

texts = [text for text, _ in corpus]
y = np.array([label for _, label in corpus])
cleaned = np.array([preprocess_text(text) for text in texts])

def make_model():
    return LogisticRegression(C=4.0, class_weight="balanced", random_state=42, max_iter=2000)

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []
y_true = []
y_pred = []
mistakes = []

for fold_number, (train_index, test_index) in enumerate(folds.split(cleaned, y), start=1):
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    model = make_model()
    model.fit(vectorizer.fit_transform(cleaned[train_index]), y[train_index])
    predicted = model.predict(vectorizer.transform(cleaned[test_index]))
    score = float((predicted == y[test_index]).mean())
    fold_scores.append(score)
    print(f"Fold {fold_number}: {score:.3f} on {len(test_index)} sentences")
    for index, guess in zip(test_index, predicted):
        y_true.append(y[index])
        y_pred.append(guess)
        if y[index] != guess:
            mistakes.append((texts[index], y[index], guess))

print(f"\nMean accuracy: {np.mean(fold_scores):.3f}")
print("\n" + classification_report(y_true, y_pred, zero_division=0))
print("Confusion matrix")
print("rows are the true label, columns follow this order:")
print(list(taxonomy))
print(confusion_matrix(y_true, y_pred, labels=list(taxonomy)))
print("\nMissed sentences:")
for sentence, true_label, guess in mistakes:
    print(f"  {true_label} -> {guess}: {sentence}")

vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
model = make_model()
model.fit(vectorizer.fit_transform(cleaned), y)
print("\nFinal assistant fitted on all", len(corpus), "labeled sentences.")


Fold 1: 0.882 on 17 sentences
Fold 2: 0.941 on 17 sentences
Fold 3: 0.941 on 17 sentences
Fold 4: 0.941 on 17 sentences
Fold 5: 1.000 on 17 sentences

Mean accuracy: 0.941

              precision    recall  f1-score   support

      advice       0.94      0.88      0.91        17
     booking       0.88      1.00      0.93        14
   emergency       0.95      1.00      0.97        18
     general       1.00      1.00      1.00         9
    location       1.00      0.86      0.92        14
  medication       0.92      0.92      0.92        13

    accuracy                           0.94        85
   macro avg       0.95      0.94      0.94        85
weighted avg       0.94      0.94      0.94        85

Confusion matrix
rows are the true label, columns follow this order:
['emergency', 'booking', 'medication', 'location', 'advice', 'general']
[[18  0  0  0  0  0]
 [ 0 14  0  0  0  0]
 [ 0  0 12  0  1  0]
 [ 0  2  0 12  0  0]
 [ 1  0  1  0 15  0]
 [ 0  0  0  0  0  9]]

Missed sentence

## Steps 7 and 8: Knowledge base and decision rules

The reply depends on the medicine or the problem, not on one fixed sentence per intent. Medicine answers are the usual label warning, not a dose made up for the person.

Rules run in this order:

1. Symptom emergencies go to 112 even if the classifier is unsure. The word `emergency` by itself does not count, so asking for the emergency department stays a location question.
2. Weather, poems, and similar topics are refused.
3. If the top probability is below 0.40, the bot says it is unsure. A blind guess among 6 intents is about 0.17, so 0.40 means the model is actually leaning somewhere.


In [8]:
NOTE = " This is general information for the assignment, not a prescription."

knowledge_base = {
    "paracetamol": (
        "Paracetamol is used for pain and fever. Side effects are uncommon at the dose on the strip. "
        "Taking more than that, or two products that both contain paracetamol, can injure the liver. "
        "For adults the usual upper limit is 4 grams in 24 hours, and lower if a doctor has already said the liver is a problem."
        + NOTE
    ),
    "ibuprofen": (
        "Ibuprofen is used for pain, fever, and swelling. Take it with food. It can cause heartburn or a stomach upset, "
        "and people with a stomach ulcer, kidney disease, or late pregnancy are usually told to avoid it. "
        "For a child the dose depends on age and weight, so follow the table on the paediatric bottle and do not use an adult tablet for a baby."
        + NOTE
    ),
    "amoxicillin": (
        "Amoxicillin is an antibiotic for some bacterial infections. It does not treat a viral cold or flu. "
        "Common side effects are diarrhoea, nausea, and a skin rash. Hives, swelling of the face, or trouble breathing can be an allergy: stop it and get help. "
        "In pregnancy, doctors often use it when an antibiotic is needed, but only the treating doctor should decide."
        + NOTE
    ),
    "aspirin": (
        "Aspirin is used for pain and, in a low dose, sometimes for the heart if a doctor has advised it. "
        "It can upset the stomach and increase bleeding. Do not give aspirin to a child or teenager with a viral fever, because of Reye's syndrome. "
        "Use the dose on the packet or the dose the doctor wrote."
        + NOTE
    ),
    "antibiotic_pregnancy": (
        "Antibiotics are not all the same in pregnancy. A penicillin such as amoxicillin is often used when it is needed. "
        "Others, such as tetracyclines, are avoided. Do not start or stop one on your own."
        + NOTE
    ),
    "multivitamin": (
        "A daily multivitamin at the dose on the bottle is fine for many people. It does not replace food. "
        "Very large amounts, especially of vitamins A, D, E, and K, can be harmful."
        + NOTE
    ),
    "medication_default": (
        "Tell me the medicine, for example paracetamol, ibuprofen, amoxicillin, or aspirin. "
        "Do not change a dose without a doctor or a pharmacist."
        + NOTE
    ),
    "burn": (
        "For a small burn, hold it under cool running water for about 20 minutes. Do not use ice, toothpaste, or butter, and do not burst a blister. "
        "See a doctor if it is large, on the face, hands, or a joint, or if it was chemical or electrical."
        + NOTE
    ),
    "bp": (
        "For high blood pressure, cut down extra salt, walk on most days, and do not stop a prescribed tablet because you feel fine. "
        "Chest pain or sudden breathlessness needs a doctor the same day."
        + NOTE
    ),
    "cholesterol": (
        "If cholesterol is high, cut down fried and fatty food and have more vegetables, fruit, dals, and oats. A daily walk helps. "
        "A statin is only for someone a doctor has put on it."
        + NOTE
    ),
    "sugar": (
        "To keep blood sugar steadier, skip sugary drinks, keep meal times regular, and take a short walk after you eat. "
        "If you already have diabetes, check the number rather than guessing from how you feel."
        + NOTE
    ),
    "cold": (
        "A common cold is usually viral. Rest, fluids, and steam or saline drops are the usual care. Paracetamol can help fever and body ache. "
        "An antibiotic does not cure a cold. See a doctor if breathing gets difficult or the fever lasts more than about three days."
        + NOTE
    ),
    "headache": (
        "For a mild headache, drink water, rest your eyes, and sleep if you are short of sleep. "
        "A sudden very severe headache, or one with a stiff neck, weakness, slurred speech, or confusion, needs emergency care."
        + NOTE
    ),
    "heart": (
        "For the heart, walk on most days, do not smoke, and keep fried food and extra salt limited. "
        "Chest pain or sudden breathlessness is not a lifestyle question. Call 112."
        + NOTE
    ),
    "tired": (
        "Being a bit tired often settles with sleep, water, and a proper meal. "
        "If it has gone on for weeks, or it comes with chest pain, breathlessness, or weight loss, get it checked."
        + NOTE
    ),
    "advice_default": (
        "For a non-urgent problem, notice whether it is getting worse, rest, and drink water. "
        "Say if it is a cold, a small burn, blood pressure, cholesterol, blood sugar, or a mild headache."
        + NOTE
    ),
    "pharmacy": (
        "I cannot see where you are, so I should not name one shop as the nearest. "
        "In Patiala, chemist shops are usually on the roads around Rajindra Hospital. Call the shop before you go. "
        "If this is an emergency, call 112 rather than searching for a chemist."
    ),
    "hospital": (
        "Rajindra Hospital in Patiala has a general emergency department and is attached to Government Medical College. "
        "I do not have your location, so I cannot say it is the closest hospital to your house. "
        "If someone is seriously ill now, call 112, or 108 for an ambulance."
    ),
    "booking": (
        "To book, shift, or cancel a visit, keep the patient's name, phone number, department, and the day you want. "
        "The hospital OPD desk does this. Morning slots usually fill before evening ones. "
        "Cancel the old slot if you shift the day. This assistant cannot confirm a real appointment."
    ),
    "general": (
        "I am a medical information assistant for this assignment. I can talk about appointments, common medicines in general terms, "
        "a hospital in Patiala, and simple home care. I am not a doctor."
    ),
}

EMERGENCY_REPLY = (
    "This sounds urgent. Call 112 now, or 108 if you need an ambulance. Do not wait for a chat reply. "
    "If you are in Patiala, the emergency department at Rajindra Hospital can take this."
)
LOW_CONFIDENCE_REPLY = (
    "I am not sure which of these you mean: an appointment, a medicine, a hospital, home care, or an emergency. "
    "Please say it in one short line."
)
OUT_OF_SCOPE_REPLY = (
    "That is outside this assistant. I only answer health questions: appointments, medicines, a hospital, home care, or an emergency."
)
CONFIDENCE_MIN = 0.40

def is_symptom_emergency(query):
    text = query.lower().replace("’", "'")
    chest = "chest" in text and any(word in text for word in ["pain", "hurt", "pressure", "tight"])
    breath = any(phrase in text for phrase in [
        "can't breathe", "cannot breathe", "cant breathe",
        "difficulty breathing", "short of breath", "shortness of breath",
    ])
    collapse = any(phrase in text for phrase in [
        "unresponsive", "not responding", "collapsed", "unconscious",
        "cannot wake", "will not wake", "won't wake",
    ])
    seizure = "seizure" in text or "fitting" in text
    stroke = (
        ("numb" in text and any(word in text for word in ["side", "arm", "face", "body"]))
        or "slurred" in text
        or "droop" in text
        or ("weak" in text and any(word in text for word in ["arm", "side", "face", "left", "right"]))
    )
    bleed = "bleed" in text and any(phrase in text for phrase in ["will not stop", "won't stop", "heavy", "lot of"])
    return chest or breath or collapse or seizure or stroke or bleed

def lookup(intent, query):
    text = query.lower()
    if intent == "medication":
        if any(name in text for name in ["paracetamol", "crocin", "dolo"]):
            return knowledge_base["paracetamol"]
        if "ibuprofen" in text:
            return knowledge_base["ibuprofen"]
        if "amoxicillin" in text:
            return knowledge_base["amoxicillin"]
        if "aspirin" in text:
            return knowledge_base["aspirin"]
        if "pregnan" in text:
            return knowledge_base["antibiotic_pregnancy"]
        if "multivitamin" in text or "vitamin" in text:
            return knowledge_base["multivitamin"]
        return knowledge_base["medication_default"]
    if intent == "advice":
        if "burn" in text:
            return knowledge_base["burn"]
        if "blood pressure" in text or "salt" in text:
            return knowledge_base["bp"]
        if "cholesterol" in text:
            return knowledge_base["cholesterol"]
        if "sugar" in text or "diabetes" in text:
            return knowledge_base["sugar"]
        if "cold" in text:
            return knowledge_base["cold"]
        if "head" in text:
            return knowledge_base["headache"]
        if "heart" in text:
            return knowledge_base["heart"]
        if "tired" in text or "tire" in text:
            return knowledge_base["tired"]
        return knowledge_base["advice_default"]
    if intent == "location":
        if "pharmac" in text or "chemist" in text:
            return knowledge_base["pharmacy"]
        return knowledge_base["hospital"]
    if intent == "booking":
        return knowledge_base["booking"]
    return knowledge_base["general"]

def analyze_query(query):
    if is_symptom_emergency(query):
        return "emergency", EMERGENCY_REPLY, "safety rule"
    if any(word in query.lower() for word in ["poem", "weather", "joke", "movie", "song", "cricket", "homework"]):
        return "unsupported", OUT_OF_SCOPE_REPLY, "out of scope"
    cleaned_query = preprocess_text(query)
    raw_tokens = re.sub(r"[^a-zA-Z\s]", " ", query.lower()).split()
    if not cleaned_query or all(word in stop_words for word in raw_tokens):
        return "unsupported", OUT_OF_SCOPE_REPLY, "no health words"
    probabilities = model.predict_proba(vectorizer.transform([cleaned_query]))[0]
    best = int(np.argmax(probabilities))
    confidence = float(probabilities[best])
    intent = model.classes_[best]
    if confidence < CONFIDENCE_MIN:
        return "unsupported", LOW_CONFIDENCE_REPLY, f"low confidence {confidence:.2f}"
    return intent, lookup(intent, query), f"model {confidence:.2f}"


## Steps 9 and 10: Checks

These lines are not copied from the corpus. They cover a booking, home care, a hospital, two medicine cautions, an emergency, a place-only question about the emergency department, a greeting, and three messages the bot should refuse.


In [10]:
test_queries = [
    "I need to schedule a consultation with a doctor",
    "What should I do for a common cold at home?",
    "How do I get to the local clinic",
    "What are the side effects of Amoxicillin",
    "Is there a hospital close to my home",
    "Please move my Friday appointment to Monday morning",
    "Can you write a poem about flowers?",
    "Uhm ok cool",
    "Help, my chest hurts and I can't breathe",
    "My father's face is drooping and his speech is slurred",
    "I am pregnant. Is it okay to take amoxicillin?",
    "How do I cool a small burn on my finger?",
    "Where is the emergency department? I am not sick, I just need the place",
    "Is ibuprofen okay for my 8 year old?",
    "What dose of aspirin should I give my child who has a fever?",
    "Good afternoon, who are you?",
    "How is the weather today",
]

labeled = {sentence for sentence, _ in corpus}
for query in test_queries:
    if query in labeled:
        raise RuntimeError("Test query was copied from the corpus: " + query)
    intent, response, how = analyze_query(query)
    print("Query:", query)
    print("Detected Intent:", intent)
    print("Decision:", how)
    print("Response:", response)
    print("-" * 60)


Query: I need to schedule a consultation with a doctor
Detected Intent: booking
Decision: model 0.65
Response: To book, shift, or cancel a visit, keep the patient's name, phone number, department, and the day you want. The hospital OPD desk does this. Morning slots usually fill before evening ones. Cancel the old slot if you shift the day. This assistant cannot confirm a real appointment.
------------------------------------------------------------
Query: What should I do for a common cold at home?
Detected Intent: advice
Decision: model 0.55
Response: A common cold is usually viral. Rest, fluids, and steam or saline drops are the usual care. Paracetamol can help fever and body ache. An antibiotic does not cure a cold. See a doctor if breathing gets difficult or the fever lasts more than about three days. This is general information for the assignment, not a prescription.
------------------------------------------------------------
Query: How do I get to the local clinic
Detected Inten